# LangChain: Fundamental

Langchain is a framework for building applications with languange models. In this tutorial, we will learn the fundamental concepts of Langchain. 

1. Benefits of Langchain
2. Basic Runnables in Langchain: Chatmodel, Prompt, Message

The benefits of LangChain are:

* Easy switch between LLM providers: LangChain provides a unified interface for different LLM providers. Langchain exposes a standard interface for key components such as models, prompts, tool calling, output parsers,and chains.Therefore, it is easy for developers to switch between providers. It is mainly supported by langchain-core and langchain-community.

* Easy to build applications with language models: combine multiple components and models into more complex applications, there’s a growing need to efficiently connect these elements into control flows that can accomplish diverse tasks. Orchestration is crucial for building such applications. It is mainly supported by langgraph.

* Easy to track and debug: LangChain provides a platform for observability and evaluations. It is mainly supported by langsmith.

## 1. Switch among LLM providers: OpenAI, Ollama, HuggingFace, OpenRouter

#### 1.1 OpenAI API

To run the following OpenAI model, you need to have the OpenAI API key. Here, we use the [environment variable](https://help.openai.com/en/articles/5112595-best-practices-for-api-key-safety) `OPENAI_API_KEY` to store the API key. You can also set the API key in the code directly.

```
import os
os.environ["OPENAI_API_KEY"] = "your_api_key"
```


In [5]:
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}")
])
llm = ChatOpenAI(model='gpt-4o-mini')
chain = prompt | llm
result = chain.invoke({"input": "Hello, what is the capital of Canada?"})
print(result.content)


The capital of Canada is Ottawa.


#### 1.2 Ollama

To run the following code, you need to install Ollama and run the ollama locally. 

**Setup Steps:**
1. Install Ollama from https://ollama.ai
2. **Start Ollama Server:**
   - **macOS**: Open the Ollama app from Applications, or it may run automatically
   - **Linux/Windows**: Run `ollama serve` in terminal
3. Pull the model: `ollama pull qwen2.5:3b`
4. Verify Ollama is running by checking: `curl http://localhost:11434/api/tags`

**Troubleshooting:**
- If you get a `ConnectError`, the Ollama server is not running
- On macOS, try opening the Ollama app from Applications folder
- Check if server is accessible: `curl http://localhost:11434/api/tags`
- If the server is running but models aren't available, pull them: `ollama pull qwen2.5:3b`


In [7]:
# Helper function to check if Ollama is running
from langchain_ollama import ChatOllama
llm_qwen = ChatOllama(model="qwen2.5:3b")
chain_qwen = prompt | llm_qwen
result_qwen = chain_qwen.invoke({"input": "Hello, what is the capital of Canada?"})
print(result_qwen.content)

The capital of Canada is Ottawa.



#### 1.3 HuggingFace

To run the following code, you need to install HuggingFace and run the HuggingFace server.

In [8]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
llm_hf = HuggingFacePipeline.from_model_id(
    model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation",
    pipeline_kwargs=dict(max_new_tokens=256, 
                         do_sample=True, 
                         temperature=0.7, 
                         top_k=50, 
                         top_p=0.95,
    ),
)
chat_model = ChatHuggingFace(llm=llm_hf)
chain_hf = prompt | chat_model.bind(skip_prompt=True)
result_hf = chain_hf.invoke({"input": "Hello, what is the capital of Canada?"})
print(result_hf.content)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use mps:0


The capital of Canada is Ottawa.


#### 1.4 OpenRouter

OpenRouter is a unified API for accessing multiple LLM providers. To use OpenRouter, you need to have an OpenRouter API key. Set it as an environment variable `OPENROUTER_API_KEY` or directly in the code:

```
import os
os.environ["OPENROUTER_API_KEY"] = "your_api_key"
```

OpenRouter allows you to access various models from different providers through a single API endpoint.


In [12]:
import os
from langchain_openai import ChatOpenAI
model_free = "tngtech/deepseek-r1t-chimera:free"
# OpenRouter uses OpenAI-compatible API, so we can use ChatOpenAI with base_url
llm_openrouter = ChatOpenAI(
    model=model_free,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)
chain_openrouter = prompt | llm_openrouter
result_openrouter = chain_openrouter.invoke({"input": "Hello, what is the capital of Canada?"})
print(result_openrouter.content)

Okay, the user is asking for the capital of Canada. I need to make sure I provide the correct information. Canada's capital is Ottawa. I should double-check to confirm. Yes, Ottawa is the capital city located in the province of Ontario. It's different from the larger cities like Toronto or Montreal, which people sometimes confuse. 

I should present the answer clearly and concisely. Maybe I can add a bit more context, like mentioning it's where the Parliament is located, to give more value. But I shouldn't make it too long unless they ask for more details. 

Wait, should I consider if they might be asking for something else? Like, maybe they meant a different Canada, but no, Canada as a country is straightforward. I don't think there's any ambiguity here. 

Also, I should keep the response friendly. Starting with "The capital of Canada is..." and then add the extra detail about Parliament. That should be good. 

Let me make sure there are no recent changes. As far as I know, Ottawa has

## 3. Direct Usage vs LangChain: A Comparison

Let's compare direct API usage with LangChain to understand the benefits. We'll show examples for OpenRouter and Ollama.


#### 3.1 Direct Usage of OpenRouter (without LangChain)


In [13]:
import os
import requests

# Direct API call to OpenRouter
def call_openrouter_direct(api_key, model, messages):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    data = {
        "model": model,
        "messages": messages
    }
    response = requests.post(url, headers=headers, json=data)
    return response.json()

# Example usage
api_key = os.getenv("OPENROUTER_API_KEY")
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello, what is the capital of Canada?"}
]

result_direct = call_openrouter_direct(api_key, model_free, messages)
print("Direct API Response:")
print(result_direct["choices"][0]["message"]["content"])


Direct API Response:
Okay, the user is asking, "Hello, what is the capital of Canada?" Let's break this down. First, they start with a greeting, which is polite, so I should mirror that in my response. The main question is about Canada's capital. 

I know the capital of Canada is Ottawa. But I should make sure I'm not mixing it up with other major cities like Toronto or Montreal, which are more well-known but not the capital. Yes, definitely Ottawa. 

I should provide a clear and concise answer. Maybe add a bit more information to be helpful, like its location or why it's the capital. But the user didn't ask for extra details, so I shouldn't overdo it. Just keep it friendly and straightforward. 

Also, considering the user might not be a native English speaker, using simple language is better. Maybe mention the province it's in, Ontario, to give a bit more context without overwhelming them. 

Wait, is there any chance the capital has changed? No, Ottawa has been the capital since 1857.

#### 3.2 Direct Usage of Ollama (without LangChain)


In [14]:
import requests

# Direct API call to Ollama
def call_ollama_direct(model, messages):
    url = "http://localhost:11434/api/chat"
    data = {
        "model": model,
        "messages": messages,
        "stream": False
    }
    response = requests.post(url, json=data)
    return response.json()

# Example usage
messages_ollama = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello, what is the capital of Canada?"}
]

result_ollama_direct = call_ollama_direct("qwen2.5:3b", messages_ollama)
print("Direct Ollama API Response:")
print(result_ollama_direct["message"]["content"])


Direct Ollama API Response:
The capital of Canada is Ottawa.


## 4. The Edge of LangChain

Now let's see why LangChain provides significant advantages over direct API usage:


#### 4.1 Unified Interface - Easy Provider Switching

With LangChain, switching between providers is as simple as changing one line of code. Notice how the same `prompt | llm` chain works across all providers:


In [15]:
import os
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

# Same prompt template works with all providers
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}")
])

# Switch providers easily - same interface!
providers = {
    "OpenAI": ChatOpenAI(model='gpt-4o-mini'),
    "OpenRouter": ChatOpenAI(
        model=model_free,
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY")
    ),
    "Ollama": ChatOllama(model="qwen2.5:3b")
}

# Same chain works for all
for name, llm in providers.items():
    try:
        chain = prompt | llm
        result = chain.invoke({"input": "What is 2+2?"})
        print(f"{name}: {result.content}")
    except Exception as e:
        print(f"{name}: Error - {str(e)}")


OpenAI: 2 + 2 equals 4.
OpenRouter: Okay, the user is asking "What is 2+2?". That seems really simple, but I should consider why they're asking this. Maybe they're just testing if I can do basic math, or perhaps they're using it as an example for something else. Let me think.

First, the answer is straightforward: 2 plus 2 equals 4. But should I just give the answer directly, or expand a bit? Since it's such a basic question, maybe they want more context. Maybe they're a young student learning math, or someone checking if I understand arithmetic.

I could explain how addition works. For example, if you have two apples and someone gives you two more, you end up with four apples. Visual examples might help. But the question is very short, so maybe they want a concise answer. I don't want to overcomplicate things if they just need the sum.

Also, I should make sure there's no trick here. Sometimes people ask simple questions with hidden meanings, but in this case, it's probably just math.

#### 4.2 Easy Composition with LCEL (LangChain Expression Language)

LangChain makes it easy to compose complex workflows. Compare the complexity:


In [17]:
from langchain_core.output_parsers import StrOutputParser

# With LangChain: Simple composition
llm = ChatOpenAI(model='gpt-4o-mini')
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}")
])
output_parser = StrOutputParser()

# Chain multiple components easily
chain = prompt | llm | output_parser

# This chain: prompt formatting -> LLM call -> output parsing
result = chain.invoke({"input": "What is the capital of France?"})
print(f"Result: {result}")
print(f"Type: {type(result)}")  # Already a string, no manual parsing needed!


Result: The capital of France is Paris.
Type: <class 'str'>


#### 4.3 Built-in Features: Streaming, Batching, Async

LangChain provides powerful features out of the box that would require significant code to implement directly:


In [18]:
# Streaming - get responses as they're generated
llm = ChatOpenAI(model='gpt-4o-mini')
prompt = ChatPromptTemplate.from_messages([
    ("human", "Count from 1 to 5, saying each number on a new line")
])
chain = prompt | llm

print("Streaming response:")
for chunk in chain.stream({"input": ""}):
    print(chunk.content, end="", flush=True)
print("\n")


Streaming response:
1  
2  
3  
4  
5  



In [19]:
# Batching - process multiple inputs efficiently
prompt = ChatPromptTemplate.from_messages([
    ("human", "What is the capital of {country}?")
])
chain = prompt | ChatOpenAI(model='gpt-4o-mini')

countries = ["France", "Japan", "Brazil", "Australia"]
results = chain.batch([{"country": c} for c in countries])

print("Batch processing results:")
for country, result in zip(countries, results):
    print(f"{country}: {result.content}")


1  
2  
3  
4  
5  


In [ ]:
# Async support - for concurrent requests
import asyncio

async def async_example():
    chain = prompt | ChatOpenAI(model='gpt-4o-mini')
    
    # Process multiple requests concurrently
    tasks = [
        chain.ainvoke({"country": "France"}),
        chain.ainvoke({"country": "Japan"}),
        chain.ainvoke({"country": "Brazil"})
    ]
    
    results = await asyncio.gather(*tasks)
    print("Async results:")
    for result in results:
        print(f"- {result.content}")


#### 4.4 Summary: Why Use LangChain?

**Without LangChain:**
- Different API formats for each provider (OpenRouter uses different structure than Ollama)
- Manual error handling and retry logic
- Manual response parsing
- Complex code for streaming, batching, async
- Hard to switch providers (requires rewriting code)
- No built-in prompt templates or message management

The same code works across OpenAI, OpenRouter, Ollama, HuggingFace, and many more providers!


## 2. Basic Runnables in LangChain

LangChain’s core abstraction is the *Runnable* interface, which provides a standard way to compose and execute language model chains. Via LCEL, runnables can be chained together using operators like (pipe) and + (combine), allowing you to build complex workflows from simple components. They support batch processing, streaming, async execution and other advanced features. The supported interfaces are as follows:
| Interface | Description |
| --- | --- |
| Invoked | A single input is transformed into an output |
| Batched | Multiple inputs are efficiently transformed into outputs |
| Streamed | Outputs are streamed as they are produced |
| Inspected | Schematic information about Runnable's input, output, and configuration can be accessed |
| Composed | Multiple Runnables can be composed to work together using the LangChain Expression Language (LCEL) to create complex pipelines |

And the major types of predefined runnables are as follows:

| Component | Input Type | Output Type |
| --- | --- | --- |
| Prompt | dictionary | PromptValue |
| ChatModel | a string, list of chat messages or a PromptValue | ChatMessage |
| LLM | a string, list of chat messages or a PromptValue | String |
| OutputParser | the output of an LLM or ChatModel | Depends on the parser |
| Retriever | a string | List of Documents |
| Tool | a string or dictionary, depending on the tool | Depends on the tool |


#### 2.1 ChatModel & LLM 

ChatModel is a wrapper around an LLM which provide a standardized way to interact with modern LLMs through a message-based interface. It should be preferred for development than Legacy LLM.

In [ ]:
from langchain_openai import OpenAI
llm = OpenAI()
response = llm.invoke("What is after Monday?")
print(response) # string format



After Monday, the next day is Tuesday.


In [10]:

from langchain_openai import ChatOpenAI
chat = ChatOpenAI()
response = chat.invoke("What is after Monday?")
print(response) # ChatMessage format
# Returns: AIMessage(content='Tuesday')

content='Tuesday' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 2, 'prompt_tokens': 12, 'total_tokens': 14, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-4c0d7a68-6458-4636-aa1a-cc1ba7a8addc-0' usage_metadata={'input_tokens': 12, 'output_tokens': 2, 'total_tokens': 14, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


#### 2.2 Prompt & Message

1. Messages are the fundamental units of communication in chat models. They represent individual pieces of a conversation and have specific roles and content. 
2. Prompts are templates that help structure how we format inputs before sending them to language models.

For message, it has four roles: Human, System, AI, and ToolMessage. 

HumanMessage: User inputs

SystemMessage: Sets behavior/context for the AI

AIMessage: Model responses

ToolMessage: Results from tool calls

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
   
messages = [
       SystemMessage(content="You are a helpful assistant"),
       HumanMessage(content="What is LangChain?")
   ]

Messages can help us to directly interact with ChatModels with the fine-grained control over conversion and specific conversion roles. And prompts can be beneficial for reusable templates, standardization inputs with variable contents.

In [17]:
from langchain_core.prompts import ChatPromptTemplate
chat_template = ChatPromptTemplate([
       ("system", "You are a helpful assistant"),
       ("user", "Tell me about {topic}")
])

In [18]:
print(chat_template.invoke({"topic": "AI"}))

messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me about AI', additional_kwargs={}, response_metadata={})]


In the above, we can see string is used to format the message. And we can also pass a list of messages to format the prompt. The example could be found as below:

In [20]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage
prompt_template = ChatPromptTemplate([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder("msgs")
])
prompt_template.invoke({"msgs": [HumanMessage(content="hi!")]})

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='hi!', additional_kwargs={}, response_metadata={})])

The following code is an example of how to use the prompt and message in LangChain to create the few-shot prompts.

In [22]:
from langchain_core.prompts import (
    FewShotChatMessagePromptTemplate,
    ChatPromptTemplate
)
examples = [
    {"input": "what is 2~2", "output": "4"},
    {"input": "what is 2~3", "output": "6"},
    {"input": "what is 4~9", "output": "36"},
    {"input": "what is 25~2", "output": "50"},
]
example_prompt = ChatPromptTemplate(
    [('human', '{input}'), ('ai', '{output}')]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    # This is a prompt template used to format each individual example.
    example_prompt=example_prompt,
)
final_prompt = ChatPromptTemplate(
    [
        ('system', 'You are a helpful AI Assistant and you should use the examples to answer the question'),
        few_shot_prompt,
        ('human', '{input}'),
    ]
)
print(final_prompt)
chain = final_prompt | llm
chain.invoke({"input": "what is 4~4?"})

input_variables=['input'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful AI Assistant and you should use the examples to answer the question'), additional_kwargs={}), FewShotChatMessagePromptTemplate(examples=[{'input': 'what is 2~2', 'output': '4'}, {'input': 'what is 2~3', 'output': '6'}, {'input': 'what is 4~9', 'output': '36'}, {'input': 'what is 25~2', 'output': '50'}], input_variables=[], input_types={}, partial_variables={}, example_prompt=ChatPromptTemplate(input_variables=['input', 'output'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=['output'], input_types={}, partial_variables={}, template='{output}'), additional_kwargs=

'\nAI: 16'